In [62]:
from pyfaidx import Fasta

In [63]:
genome = Fasta(
    "../data/reference/hg38.fa"
)

In [64]:
genome

Fasta("../data/reference/hg38.fa")

In [65]:
sequence = genome["chr1"][1000:1100].seq

print(sequence)

NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN


In [66]:
from pybigtools import open

bb = open(
    "../data/raw/ENCFF106WBX.bigBed"
)

In [67]:
print(bb.chroms())

{'chr1': 248956422, 'chr10': 133797422, 'chr11': 135086622, 'chr12': 133275309, 'chr13': 114364328, 'chr14': 107043718, 'chr15': 101991189, 'chr16': 90338345, 'chr17': 83257441, 'chr18': 80373285, 'chr19': 58617616, 'chr2': 242193529, 'chr20': 64444167, 'chr21': 46709983, 'chr22': 50818468, 'chr3': 198295559, 'chr4': 190214555, 'chr5': 181538259, 'chr6': 170805979, 'chr7': 159345973, 'chr8': 145138636, 'chr9': 138394717, 'chrX': 156040895, 'chrY': 57227415}


In [68]:
chrom_size = bb.chroms()["chr1"]

records = list(
    bb.records(
        "chr1",
        0,
        chrom_size
    )
)

print(len(records))
print(records[:5])

10526
[(778479, 779226, '.', '1000', '.', '26.93655', '713.72321', '709.58307', '299'), (778479, 779226, '.', '1000', '.', '3.62514', '25.43981', '23.19857', '647'), (817132, 817914, '.', '1000', '.', '6.20372', '70.13375', '67.57131', '304'), (826649, 828088, '.', '1000', '.', '11.76791', '588.28070', '584.43811', '940'), (826649, 828088, '.', '1000', '.', '2.16284', '17.48284', '15.34591', '400')]


In [69]:
first_record = records[0]

print(first_record)
"""structure is: (
  start,
  end,
  name,
  score,
  strand,
  signalValue,
  pValue,
  qValue,
  peak
)"""
print(type(first_record))

(778479, 779226, '.', '1000', '.', '26.93655', '713.72321', '709.58307', '299')
<class 'tuple'>


In [70]:
import pandas as pd

rows = []

for record in records:

    start, end, name, score, strand, signalValue, pValue, qValue, peak = record

    rows.append({
        "chrom": "chr1",
        "start": start,
        "end": end,
        "signalValue": float(signalValue),
        "pValue": float(pValue),
        "qValue": float(qValue)
    })

df = pd.DataFrame(rows)

print(df.head())

  chrom   start     end  signalValue     pValue     qValue
0  chr1  778479  779226     26.93655  713.72321  709.58307
1  chr1  778479  779226      3.62514   25.43981   23.19857
2  chr1  817132  817914      6.20372   70.13375   67.57131
3  chr1  826649  828088     11.76791  588.28070  584.43811
4  chr1  826649  828088      2.16284   17.48284   15.34591


In [71]:
row = df.iloc[0]

sequence = genome[
    row["chrom"]
][
    row["start"]:row["end"]
].seq

print(sequence[:100])
print(len(sequence))

CACTGTGCTGGGTCTGAATTTTTCAGCTACCCTTCTTCAGCCGGCAACACACAGAACCTGGCGGGGAGGTCACTCTTACCAGTCCCCACTCTGATGAGAA
747


In [72]:
WINDOW_SIZE = 200

def extract_fixed_window(
    genome,
    chrom,
    start,
    peak_offset,
    window_size=WINDOW_SIZE
):

    summit = start + peak_offset

    half_window = window_size // 2

    window_start = summit - half_window
    window_end = summit + half_window

    sequence = genome[
        chrom
    ][
        window_start:window_end
    ].seq.upper()

    return sequence

In [82]:
positive_sequences = []

chromosomes = [
    "chr1",
    "chr2",
    "chr3"
]

for chrom in chromosomes:

    print(f"Processing {chrom}")

    chrom_size = bb.chroms()[chrom]

    records = list(
        bb.records(
            chrom,
            0,
            chrom_size
        )
    )

    for record in records:

        (
            start,
            end,
            name,
            score,
            strand,
            signalValue,
            pValue,
            qValue,
            peak
        ) = record

        sequence = extract_fixed_window(
            genome=genome,
            chrom=chrom,
            start=start,
            peak_offset=int(peak)
        )

        if len(sequence) != 200:
            continue

        positive_sequences.append({
            "chrom": chrom,
            "start": start,
            "end": end,
            "sequence": sequence,
            "label": 1
        })

Processing chr1
Processing chr2
Processing chr3


In [83]:
positive_df = pd.DataFrame(
    positive_sequences
)

print(positive_df.head())
print(len(positive_df))

  chrom   start     end                                           sequence  \
0  chr1  778479  779226  CCGCCCTTGTGACGTCACGGAAGGCGCACCCTTGTGACGTCACAGG...   
1  chr1  778479  779226  CAGAAGCTCCTCAATGGCCAGCGCCAGCTGCAGCCCCGGCCGCCCA...   
2  chr1  817132  817914  GAAAAGGGAAGAATGCAAAAGTCAAAGACACGTCACCCTCCTTGAG...   
3  chr1  826649  828088  GGACCTTGGCTCCGGATAATCCGTTTCCGGGTCAACAAAAAACGTC...   
4  chr1  826649  828088  GGGAGGCGATTGTGCAGAAGCACGAGGGTTGTTACAGGATCGGGCA...   

   label  
0      1  
1      1  
2      1  
3      1  
4      1  
24708


In [84]:
print(positive_df.iloc[0]["sequence"])
print(len(positive_df.iloc[0]["sequence"]))

CCGCCCTTGTGACGTCACGGAAGGCGCACCCTTGTGACGTCACAGGGGACTACCACTCACGCAGAGCCAATCAGAACTCGCGGTGGGGGCTGCTGGTTCTTCCAGGAGCGCGCATGAGCGGACGCTGCCTACTGGTGGCCGGGCGGGATGTAACCGGCTGCTGAGCTGGCAGTTCTGTGTCGCTAGGCTTCTGCCCGGCC
200


In [85]:
from collections import Counter

sequence = positive_df.iloc[0]["sequence"]

print(Counter(sequence))

Counter({'G': 69, 'C': 62, 'T': 38, 'A': 31})


In [86]:
peak_intervals = {}

for chrom in chromosomes:

    peak_intervals[chrom] = []

    chrom_size = bb.chroms()[chrom]

    records = list(
        bb.records(
            chrom,
            0,
            chrom_size
        )
    )

    for record in records:

        start = record[0]
        end = record[1]

        peak_intervals[chrom].append(
            (start, end)
        )

In [87]:
def overlaps_peak(
    chrom,
    query_start,
    query_end,
    peak_intervals
):

    for peak_start, peak_end in peak_intervals[chrom]:

        if (
            query_start < peak_end
            and
            query_end > peak_start
        ):
            return True

    return False

In [88]:
negative_sequences = []

while len(negative_sequences) < len(positive_sequences):

    chrom = random.choice(chromosomes)

    chrom_size = bb.chroms()[chrom]

    random_start = random.randint(
        0,
        chrom_size - 200
    )

    random_end = random_start + 200

    if overlaps_peak(
        chrom,
        random_start,
        random_end,
        peak_intervals
    ):
        continue

    sequence = genome[
        chrom
    ][
        random_start:random_end
    ].seq.upper()

    if len(sequence) != 200:
        continue

    negative_sequences.append({
        "chrom": chrom,
        "start": start,
        "end": end,
        "sequence": sequence,
        "label": 0
    })

In [89]:
negative_df = pd.DataFrame(
    negative_sequences
)

print(negative_df.head())
print(len(negative_df))

  chrom      start        end  \
0  chr2  198157266  198157656   
1  chr3  198157266  198157656   
2  chr1  198157266  198157656   
3  chr3  198157266  198157656   
4  chr1  198157266  198157656   

                                            sequence  label  
0  AGTTTCATGCCTCTCAGTGGTTTTTCCTTGGAATTTCAGGGAATAG...      0  
1  CACTTAGCCTTTCATAAACATTGTACTCTAGTAGAAGTTAATTTGA...      0  
2  TCCAATCATGGTGGAGTGGATGTGAACTTCCTCCAGAGTATCTCCA...      0  
3  AATGAAAGCAGATAAGAAAAGAAAGGCATTTAAATAAATGCCCAAG...      0  
4  NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...      0  
24708


In [90]:
dataset_df = pd.concat(
    [positive_df, negative_df],
    ignore_index=True
)

dataset_df = dataset_df.sample(
    frac=1
).reset_index(drop=True)

print(dataset_df.head())

  chrom      start        end  \
0  chr2  198157266  198157656   
1  chr1  164767394  164767755   
2  chr1  198157266  198157656   
3  chr3  190345021  190345841   
4  chr2  198157266  198157656   

                                            sequence  label  
0  ATGTACAAAAACACTAGAATCGGCCAGGCGTGATGGCTCACACCTG...      0  
1  CTGGAGGCATATACATGTGCGGGGCCCGGAGAGGAGGAGTGGACAG...      1  
2  CTTGTTAGACTAGCCCTTATCTTCCTAGTCACTTCTCTACCCAATT...      0  
3  CTAGAAAGACTCTTTGATGAGGAAAAGGAATTCTGGTTCTAGTTTT...      1  
4  CTGGGCCCTGCCCCCAGGGAAGCCTCTGTCTCCAGCGGACCCCTCC...      0  


In [91]:
dataset_df.to_csv(
    "../data/processed/liver_accessibility_gc_coords.csv",
    index=False
)

In [81]:
negative_sequences[0]

{'chrom': 'chr1',
 'sequence': 'TAGCTGGGATTACAGGCACGTGTCACCACGCCCAGCTAATTTTGTACTTTTAGTAGAGATGAGTTTTCTCCATGTTGGTCAGGCTGGTCTCAAACTCCCGACTCAGGTGATCCACCCACCTCGGCCTCCCAAAGTGCTAGGATTACAGGCATGAGCCACCACATAAGCCTCTTTTTAAAAAAATTATTAGGTAAATATAG',
 'label': 0}